# Análisis de Sentimientos - Reviews de Olist Store

Notebook en construcción: preparación de datos, entrenamiento y evaluación (positivo vs. negativo).

## 1. Carga del dataset

In [ ]:
import pandas as pd

df_reviews = pd.read_csv("./datasets/olist_order_reviews_dataset.csv")
print(f"Dataset cargado correctamente: {len(df_reviews)} filas")

## 2. Limpieza y texto completo

In [ ]:
import numpy as np

# Solo con título y/o mensaje
df_text = df_reviews.dropna(
    subset=["review_comment_title", "review_comment_message"], how="all"
).copy()

# Sin neutras (3 estrellas) y target binario: 1 = 4-5 estrellas
df_filtered = df_text[df_text["review_score"] != 3].copy()
df_filtered["target"] = (df_filtered["review_score"] >= 4).astype(int)

# Orden temporal para split sin fuga
df_sorted = df_filtered.sort_values(by="review_creation_date").reset_index(drop=True)

# Une título + mensaje: quita puntuación final, une con ". " y normaliza espacios
title = (
    df_sorted["review_comment_title"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.rstrip(".!?:;")
)

message = df_sorted["review_comment_message"].fillna("").astype(str).str.strip()
separador = np.where((title != "") & (message != ""), ". ", "")
df_sorted["review_text_full"] = (
    (title + separador + message)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

print(
    f"Total de comentarios con títulos y/o mensajes: {len(df_sorted['review_text_full'])}"
)

# Features / target
X = df_sorted["review_text_full"]
y = df_sorted["target"]

## 3. División Temporal (Train/Test Split)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=False)

print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Muestras de evaluación: {len(X_test)}")

## 4. Entrenamiento de Pipelines (Naive Bayes vs Linear SVM)

In [ ]:
import unicodedata

import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

# Descargar stopwords de NLTK
nltk.download("stopwords", quiet=True)


# Normalizar acentos de las stopwords para alinearlo con strip_accents='unicode'
def remove_accents(s):
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


stop_words = {remove_accents(w) for w in stopwords.words("portuguese")}
negaciones = {
    "nao",
    "nunca",
    "jamais",
    "nenhum",
    "nenhuma",
    "nem",
    "nada",
    "sem",
    "tampouco",
}
stop_custom = list(stop_words - negaciones)

# Definición de modelos a comparar
modelos = {
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42, class_weight="balanced"),
}

pipelines = {}
resultados: list[dict] = []

for nombre, clf in modelos.items():
    pipe = Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    strip_accents="unicode",
                    ngram_range=(1, 2),
                    min_df=2,
                    stop_words=stop_custom,
                ),
            ),
            ("clf", clf),
        ]
    )

    pipe.fit(X_train, y_train)  # tfidf se ajusta solo sobre X_train (sin leakage)
    pipelines[nombre] = pipe

    y_pred = pipe.predict(X_test)

    # NB tiene predict_proba, SVM tiene decision_function -> unificamos el "score"
    if hasattr(pipe.named_steps["clf"], "predict_proba"):
        score = pipe.predict_proba(X_test)[:, 1]
    else:
        score = pipe.decision_function(X_test)

    reporte = classification_report(
        y_test,
        y_pred,
        target_names=["Negativo", "Positivo"],
        digits=4,
        output_dict=True,
    )
    assert isinstance(reporte, dict)

    auc = roc_auc_score(y_test, score)

    print(f"\n{'=' * 69}\n  EVALUACIÓN {nombre.upper()}\n{'=' * 69}")
    print(
        classification_report(
            y_test, y_pred, target_names=["Negativo", "Positivo"], digits=4
        )
    )
    print(f"ROC-AUC Score: {auc:.4f}")

    resultados.append(
        {
            "Modelo": nombre,
            "Accuracy": reporte["accuracy"],
            "F1 Negativo": reporte["Negativo"]["f1-score"],
            "Recall Negativo": reporte["Negativo"]["recall"],
            "F1 Positivo": reporte["Positivo"]["f1-score"],
            "ROC-AUC": auc,
        }
    )
print("="*69)

df_comparacion = pd.DataFrame(resultados).set_index("Modelo")
display(df_comparacion)